In [1]:
import os, pickle, tempfile
import pandas as pd

# Data

In [2]:
extra_set_path = "../../training_data/8.Apos/Extra_set"

In [3]:
with open(f"{extra_set_path}/apos.pkl", "rb") as f:
    apos = pickle.load(f)

len(apos), apos

(9,
 {'8jp0': {'8sgj': ['A']},
  '8uk6': {'AF-A0A1D8PQM9-F1': ['A']},
  '8f4s': {'7l6r': ['A']},
  '8qni': {'8vw5': ['A']},
  '7gqu': {'6yhr': ['A']},
  '7yg5': {'7xlq': ['A']},
  '8aq6': {'5b0u': ['A']},
  '8v81': {'5uak': ['A']},
  '9dnm': {'4jqi': ['A']}})

In [4]:
with open(f"{extra_set_path}/pockets_features.pkl", "rb") as f:
    news = pickle.load(f)

len(news), news

(9,
 {'8sgj':    Pockets                                              Label      FPocket  \
         pdb    pocket nres site_in_pocket pocket_in_site label Pocket Score   
  0     8sgj  pocket16   12        0.00000       0.000000     0      -0.0342   
  1     8sgj  pocket28   14        0.00000       0.000000     0      -0.1447   
  2     8sgj  pocket18   13        0.03125       0.076923     0      -0.0508   
  3     8sgj  pocket11   19        0.00000       0.000000     0       0.0417   
  4     8sgj  pocket25   13        0.00000       0.000000     0      -0.1152   
  5     8sgj  pocket15    9        0.03125       0.111111     0      -0.0332   
  6     8sgj   pocket7    8        0.03125       0.125000     0       0.1316   
  7     8sgj  pocket22   10        0.00000       0.000000     0      -0.0848   
  8     8sgj  pocket13    8        0.00000       0.000000     0       0.0031   
  9     8sgj  pocket23   40        0.00000       0.000000     0      -0.1057   
  10    8sgj  pocket14    9 

In [5]:
with open(f"{extra_set_path}/apos_sites.pkl", "rb") as f:
    news_sites =  {k: [{"site": s["site"]} for s in v.values()] for k,v in pickle.load(f).items() if k in news}

len(news_sites), news_sites

(9,
 {'8sgj': [{'site':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
    0            GLU             A               1          132                 ?   
    1            THR             A               1          133                 ?   
    2            VAL             A               1          134                 ?   
    3            SER             A               1          135                 ?   
    4            LEU             A               1          137                 ?   
    5            THR             A               1          138                 ?   
    6            ALA             A               1          141                 ?   
    7            HIS             A               1          200                 ?   
    8            VAL             A               1          203                 ?   
    9            VAL             A               1          206                 ?   
    10           THR             A           

In [6]:
assert all(len(sites) == 1 for sites in news_sites.values()), "Not all apos have a single annotated site"

In [7]:
models = pd.read_pickle("models_lenient_labelling.pkl")

models

{'model5': {'results': {'4jqi': {'pocket16': {'prob': 0.0001226096646860242,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket18': {'prob': 0.003527346532791853,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket11': {'prob': 4.818243542104028e-05,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket15': {'prob': 1.1008408629109567e-09,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket7': {'prob': 0.0006957116420380771,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket13': {'prob': 0.09678139537572861,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.

In [8]:
colors = {
    "orange": "#D55E00".lower(),
    "green": "#009E73".lower(),
    "blue": "#0072B2".lower()
}

# Labelling

In [9]:
# Percentage of residues of "one" in "other"
get_overlap = lambda one, other: (
    len( one.merge(other) ) / len(one)
)

get_overlaps = lambda pdb, pocketd: {
    name: get_overlap(*one_in_other) 
        for site in news_sites[pdb] 
            for name, one_in_other in (
                ("pocket_in_site", (pocketd["residues"], site["site"])),
                ("site_in_pocket", (site["site"], pocketd["residues"])),
            )
}

def get_label(overlaps, site_in_pocket=None, pocket_in_site=None):
    assert not (site_in_pocket==None and pocket_in_site==None)
    
    if site_in_pocket is None:
        return int( overlaps["pocket_in_site"] >= pocket_in_site )
    if pocket_in_site is None:
        return int( overlaps["site_in_pocket"] >= site_in_pocket )
    return int( overlaps["site_in_pocket"] >= site_in_pocket or overlaps["pocket_in_site"] >= pocket_in_site )

In [10]:
# def label_results(resultsd, site_in_pocket=0.65, pocket_in_site=None, prob_key=None):
#     return pd.DataFrame((
#         {
#             "pdb": pdb,
#             "pocket": pocket,
#             **{"prob": pocketd[prob_key] for prob_key in (prob_key,) if prob_key is not None},
#             "pred": pocketd["pred"],
#             "label": get_label(overlaps, site_in_pocket, pocket_in_site),
#             "max_overlap": max(overlaps.values()),
#             **overlaps,
#         }
#         for pdb, pockets in resultsd.items()
#         for pocket, pocketd in pockets.items()
#         for overlaps in (get_overlaps(pdb, pocketd),)
#     )).sort_values("max_overlap", ascending=False)

def label_results(resultsd, site_in_pocket=0.65, pocket_in_site=None, prob_key=None):
    df = pd.DataFrame((
        {
            "pdb": pdb,
            "pocket": pocket,
            **{"prob": pocketd[prob_key] for prob_key in (prob_key,) if prob_key is not None},
            "pred": pocketd["pred"],
            "label": get_label(overlaps, site_in_pocket, pocket_in_site),
            "max_overlap": max(overlaps.values()),
            **overlaps,
        }
        for pdb, pockets in resultsd.items()
        for pocket, pocketd in pockets.items()
        for overlaps in (get_overlaps(pdb, pocketd),)
    ))
    if prob_key is not None:
        df["pred"] = (
            # Start from a Series where each value/row (sample/pocket) is the total number of pos. labels on its PDB
            df.groupby("pdb")["label"].transform("sum")
            # Then subtract this "total num. of pos. in a PDB" by the rank of each pocket in a PDB, sorted by the probability
            .sub(df.groupby("pdb")["prob"].rank(method="first", ascending=False))
            # If the subtraction is positive or 0 it means that the pocket is in the topX and will be assigned 1
            >= 0
        ).astype(int)
    
    return df.sort_values("max_overlap", ascending=False)

In [11]:
# def label_our_results(resultsd):#, site_in_pocket=0.65, pocket_in_site=None):
#     return pd.DataFrame((
#         {
#             "pdb": pdb,
#             "pocket": pocket,
#             "prob": pocketd["prob"],
#             "pred": pocketd["pred"],
#             "pred_top1": int( pocketd["prob"] == pdb_maxprob ),
#             "label": pocketd["label"],
#             "max_overlap": max(overlaps.values()),
#             **overlaps,
#         }
#         for pdb, pockets in resultsd.items()
#         for pdb_maxprob in (max(pktd["prob"] for pktd in pockets.values()),)
#         for pocket, pocketd in pockets.items()
#         for overlaps in ({k: pocketd[k] for k in ["pocket_in_site", "site_in_pocket"]},)
#     )).sort_values("max_overlap", ascending=False)

def label_our_results(resultsd, site_in_pocket=0.65, pocket_in_site=None):
    df = pd.DataFrame((
        {
            "pdb": pdb,
            "pocket": pocket,
            "prob": pocketd["prob"],
            "label": get_label(overlaps, site_in_pocket, pocket_in_site),
            "max_overlap": max(overlaps.values()),
            **overlaps,
        }
        for pdb, pockets in resultsd.items()
        for pocket, pocketd in pockets.items()
        for overlaps in ({k: pocketd[k] for k in ["pocket_in_site", "site_in_pocket"]},)
    ))
    df["pred"] = (
        # Start from a Series where each value/row (sample/pocket) is the total number of pos. labels on its PDB
        df.groupby("pdb")["label"].transform("sum")
        # Then subtract this "total num. of pos. in a PDB" by the rank of each pocket in a PDB, sorted by the probability
        .sub(df.groupby("pdb")["prob"].rank(method="first", ascending=False))
        # If the subtraction is positive or 0 it means that the pocket is in the topX and will be assigned 1
        >= 0
    ).astype(int)

    return df.sort_values("max_overlap", ascending=False)

In [12]:
for model, modeld in models.items():
    if model != "model5":
        models[model]["labelled"] = label_results(
            modeld["results"], 
            **modeld["labelling"], 
            prob_key=modeld["prob_key"]
        )
    else:
        models[model]["labelled"] = label_our_results(
            modeld["results"], 
            **modeld["labelling"]
        )

# Pocket functions

In [13]:
import sys

sys.path.append("../../training_data")

In [14]:
from utils.utils import Cif, CifFileWriter
from utils.pocket_utils import Pocket

In [15]:
from biotite.structure.io.pdb import PDBFile

def get_pdb_atoms(f):
    atom_array = PDBFile.read(f).get_structure()
    return pd.DataFrame({
            "auth_asym_id": atom_array.chain_id,
            "auth_seq_id": atom_array.res_id,
            "auth_comp_id": atom_array.res_name,
            "auth_atom_id": atom_array.atom_name,
            "type_symbol": atom_array.element,
            "Cartn_x": atom_array.coord[0][:, 0],
            "Cartn_y": atom_array.coord[0][:, 1],
            "Cartn_z": atom_array.coord[0][:, 2],
            "pdbx_PDB_ins_code": (ic or '?' for ic in atom_array.ins_code)
        }, dtype=str)

In [16]:
def get_pocket(pocket_atoms, res_id, color):
    pocket_atoms["label_entity_id"] = '99'
    return {
        "pocketn": res_id,
        "atoms": pocket_atoms, 
        "representation": {
            "selection": [{'label_entity_id': '99', "auth_asym_id": pocket_atoms.auth_asym_id.unique().item(), 'auth_seq_id': int(res_id)}], 
            'color': colors[color]
        }
    }

In [17]:
def get_our_pocket(pdb, pocket, color):
    pocketn = pocket.replace('pocket', '')
    # pocket_atoms = (
    #     Cif(pdb, f"{extra_set_path}/pockets/{pdb}/{pdb}_out/{pdb}_out.cif", name=f"{pdb}_out")
    #     .atoms
    #     .query(f"label_comp_id == 'STP' and label_seq_id == '{pocketn}'")
    # )

    return {
        "pocketsf": f"{extra_set_path}/pockets/{pdb}/{pdb}_out/{pdb}_out.cif",
        "pocketn": pocketn,
        "pocket_sel": [{"label_comp_id": 'STP', "label_seq_id": int(pocketn)},],
        "color": colors[color]
    }#get_pocket(pocket_atoms, pocketn, color)
# {
#         "atoms": pocket_atoms, 
#         "representation": {
#             "selection": {'label_entity_id': '99', "auth_asym_id": pocket_atoms.auth_asym_id.unique().item(), 'auth_seq_id': int(pocketn)}, 
#             'color': colors[color]
#         }
#     }

models["model5"]["pocketf"] = get_our_pocket

In [18]:
def get_allositepro_pocket(pdb, pocket, color):
    resultsf = next(f for f in os.listdir(f"AllositePro/{pdb}") if f.endswith("_download"))
    # pockets are 0-indexed but residue numbers start at 1
    # also pocket0 can be residue ID 2, so all pockets auth_seq_id will be sorted and then the relevant residue id taken with the 0-index pocket number
    pockets_atoms = (
        get_pdb_atoms(f"AllositePro/{pdb}/{resultsf}/{resultsf.replace('_download', '')}.pdb")
        .query(f"auth_comp_id == 'STP'")
    )
    pocketn = sorted(pockets_atoms.auth_seq_id.unique())[ int(pocket.replace('pocket', '')) ]
    # pocket_atoms = pockets_atoms.query(f"auth_seq_id == '{pocketn}'")
    
    return {
        "pocketsf": f"AllositePro/{pdb}/{resultsf}/{resultsf.replace('_download', '')}.pdb",
        "pocketn": pocketn,
        "pocket_sel": [{"auth_comp_id": 'STP', "auth_seq_id": int(pocketn)},],
        "color": colors[color]
    }#get_pocket(pocket_atoms, pocketn, color)

models["allositepro"]["pocketf"] = get_allositepro_pocket

In [19]:
from functools import partial

In [20]:
# def get_passer_pocket(pdb, pocket, color, model):
#     pocketn = pocket.replace('pocket', '')
#     # pocket_atoms = (
#     #     get_pdb_atoms(f"PASSer/{model}/{pdb}/{pdb}_out.pdb")
#     #     .query(f"auth_comp_id == 'STP' and auth_seq_id == '{pocketn}'")
#     # )
    
#     return {
#         "pocketsf": f"PASSer/{model}/{pdb}/{pdb}_out.pdb",
#         "pocketn": pocketn,
#         "pocket_sel": [{"auth_comp_id": 'STP', "auth_seq_id": int(pocketn)},],
#         "color": colors[color]
#     }#get_pocket(pocket_atoms, pocketn, color)

# models["passer_ensemble"]["pocketf"] = partial(get_passer_pocket, model="ensemble")
# models["passer_automl"]["pocketf"] = partial(get_passer_pocket, model="automl")
# models["passer_rank"]["pocketf"] = partial(get_passer_pocket, model="rank")

In [21]:
def get_allo_pocket(pdb, pocket, color):
    if "AF" not in pdb:
        resultsf = next(f for f in os.listdir(f"ALLO/{pdb}") if f.startswith(f"{pdb}pdb") and f.endswith("_desc.txt")).replace("_desc.txt", "")
    else:
        afpdb = pdb.lower().replace("-", "")
        resultsf = next(f for f in os.listdir(f"ALLO/{pdb}") if f.startswith(f"{afpdb}pdb") and f.endswith("_desc.txt")).replace("_desc.txt", "")
    pocket_atoms = (
        get_pdb_atoms(f"ALLO/{pdb}/pockets/{resultsf}_{pocket}_res.pdb")
    )
    
    # pocket_atoms["auth_asym_id"] = 'ZZZ'
    # pocket_atoms["label_entity_id"] = '99'
    return {
        "pocketsf": f"ALLO/{pdb}/pockets/{resultsf}_{pocket}_res.pdb",
        "pocketn": pocket,
        "pocket_sel": [
            {"auth_asym_id": atom['auth_asym_id'], "auth_seq_id": int(atom['auth_seq_id']), "auth_atom_id": atom['auth_atom_id']}
            for i, atom in pocket_atoms.query("auth_atom_id not in ['CA', 'C', 'O', 'N', 'CB']").iterrows()
        ],
        "color": colors[color]
    }
    #{
    #     "pocketn": pocket,
    #     "atoms": pocket_atoms.query("auth_atom_id not in ['CA', 'C', 'O', 'N', 'CB']"), 
    #     "representation": {
    #         "selection": [
    #             {"auth_asym_id": "ZZZ", 'auth_seq_id': int(res)}
    #             for res in pocket_atoms["auth_seq_id"].unique()
    #         ],
    #         'color': colors[color]
    #     }
    # }

models["allo"]["pocketf"] = get_allo_pocket

In [22]:
def get_mefallosite_pocket(pdb, pocket, color):
    pocketn = pocket.replace('pocket', '')
    # pocketsf = f"MEF-AlloSite/MEF-AlloSite/data/test/{pdb.upper()}_cleaned_out/{pdb.upper()}_cleaned_out.pdb"
    # pocket_atoms = (
    #     get_pdb_atoms(pocketsf)
    #     .query(f"auth_comp_id == 'STP' and auth_seq_id == '{pocketn}'")
    # )
    
    # pocket_atoms["label_entity_id"] = '99'
    return {
        "pocketsf": f"MEF-AlloSite/MEF-AlloSite/data/test/{pdb.upper()}_cleaned_out/{pdb.upper()}_cleaned_out.pdb",
        "pocketn": pocketn,
        "pocket_sel": [{"auth_comp_id": 'STP', "auth_seq_id": int(pocketn)},],
        "color": colors[color]
    }
    # {
    #     # "number": int(pocketn), 
    #     "atoms": pocket_atoms, 
    #     "representation": [{
    #         'entity_id': '99', "auth_asym_id": pocket_atoms.auth_asym_id.unique().item(), 'auth_residue_number': int(pocketn), 'representation': 'molecular-surface', 'representationColor': colors[color]
    #     },]
    #     # "color": colors[color]
    # }

models["mefallosite"]["pocketf"] = get_mefallosite_pocket

# View functions

In [23]:
import molviewspec as mvs
import json
from pathlib import Path

In [55]:
import numpy as np

In [171]:
# ass_fields_list = ["_pdbx_struct_assembly", "_pdbx_struct_assembly_gen", "_pdbx_struct_oper_list"]

def view_pockets(pdb, model, pockets:list):
    cif = Cif(pdb, f"{extra_set_path}/origcifs/{pdb}_updated.cif.gz")

    # minimal_elements = lambda element="label_asym_id": site["site"][element].unique().tolist() + site["mod"][element].unique().tolist()

    site = news_sites[pdb][0]
    if pdb == "8sgj":
        site["site"] = site["site"].loc[lambda x: x["label_seq_id"].astype(int) < 800]
    # atoms = cif.atoms.query(f"label_asym_id in {minimal_elements('label_asym_id')}")

    # # Fake entity data
    # entities = pd.concat((
    #     pd.DataFrame(cif.cif.data["_entity"], dtype=str),#.query(f"id in {minimal_elements('label_entity_id')}"),
    #     pd.DataFrame([{"id": "99", "type": "branched", "pdbx_description": "pockets"}]) # Fake the pockets as carbohydrates to manage their representation
    # )).fillna(".")

    entities = pd.DataFrame(cif.cif.data["_entity"], dtype=str)

    # columns = list( set.intersection( *map(set, (pocket_atoms["atoms"].columns for pocket_atoms in pockets)) ) )
    # atoms = pd.concat((
    #     cif.atoms[columns],
    #     *(pocket_atoms["atoms"][columns] for pocket_atoms in pockets)
    # ))

    # with tempfile.NamedTemporaryFile("w+", suffix=".cif") as f:
    #     writer = CifFileWriter(f.name)
    #     writer.write({cif.entry_id.upper(): {
    #         "_entity": entities.to_dict(orient="list"),
    #         "_atom_site": atoms.to_dict(orient="list"),
    #         # **{k: cif.cif.data[k] for k in ass_fields_list}
    #     }})
    #     combined = Cif(pdb, filename=f.name)
    #     combined.cif.data # to cache it while 'f' exists

    
    builder = mvs.create_builder()
    if "transformation" in custom_vizs[pdb]["camera"]:
        M = np.array(custom_vizs[pdb]["camera"]["transformation"]).reshape(4,4,order="F")
        rot, trans = M[:3,:3].flatten(order="F").tolist(), M[:3,3].tolist()
        structure = (
            builder.download(url=f"{pdb}.cif")
            .parse(format="mmcif")
            .model_structure()
            .transform(rotation=rot, translation=trans)
        )
    else:
        structure = (
            builder.download(url=f"{pdb}.cif")
            .parse(format="mmcif")
            .model_structure()
        )

    assets = {f'{pdb}.cif': cif.cif.text.encode()}


    if custom_vizs[pdb]["camera"]["camera"]:
        builder.camera(
            **eval(
                "dict("
                + custom_vizs[pdb]["camera"]["camera"].strip()[1:-1].replace(":", "=") 
                + ")"
            )
        )
    if model in custom_vizs[pdb]["camera"]:
        builder.camera(
            **eval(
                "dict("
                + custom_vizs[pdb]["camera"][model].strip()[1:-1].replace(":", "=") 
                + ")"
            )
        )

    # Protein and site
    for auth_asym_id, siteres in site["site"].groupby("auth_asym_id"):
        protein = (
            structure
            .component(selector=[
                mvs.ComponentExpression(**sel)
                for sel in custom_vizs[pdb]["selections"].get(f"protein_{auth_asym_id}", [{"auth_asym_id": auth_asym_id},])
            ])
            .representation(type="cartoon", custom={"molstar_representation_params": {"ignoreLight": True}})#, custom={"ignoreLight": True})
            .color(color='#DADADA')
        )
        annos = f'annos_chain_{auth_asym_id}.json'
        assets[annos] = json.dumps(
            [
                {
                    'auth_asym_id': r["auth_asym_id"], 'auth_seq_id': int(r["auth_seq_id"]), 
                    'color': 'black' # 'pdbx_PDB_ins_code': r["pdbx_PDB_ins_code"],
                }
                for i, r in siteres.iterrows()
            ]
        ).encode()
        protein.color_from_uri(uri=annos, format='json', schema='all_atomic')
            
    # Ligands
    if not custom_vizs[pdb]["ligands"]["none"]:
        for entity_id in entities.query("type == 'non-polymer'").id.unique():
            # if entity_id not in site["mod"].label_entity_id.unique():
            (
                structure
                .component(selector=mvs.ComponentExpression(label_entity_id=entity_id))
                .representation(type="ball_and_stick", custom={"molstar_representation_params": {"ignoreLight": True}})
                .color(color="white")
            )
    if model in custom_vizs[pdb]["ligands"]:
        for lig in custom_vizs[pdb]["ligands"][model]:
            (
                structure
                .component(selector=mvs.ComponentExpression(**lig))
                .representation(
                    type="ball_and_stick", 
                    size_factor=custom_vizs[pdb]["ligands"].get("size_factor", 0.6),
                    custom={"molstar_representation_params": {"ignoreLight": True}})
                .color(
                    custom={
                        "molstar_color_theme_name": "element-symbol",
                        "molstar_color_theme_params": {
                            "carbonColor": { "name": "uniform", "params": {"value": int("#808080".replace('#', ''), 16)} }
                        }
                    }
                )
                # .opacity(opacity=0.9)
            )
    
    # # Modulator
    # # for entity_id in entities.query("type == 'non-polymer'").id.unique():
    # for entity_id in site["mod"].label_entity_id.unique():
    #     (
    #         structure
    #         .component(selector=mvs.ComponentExpression(label_entity_id=entity_id))
    #         .representation(
    #             type="ball_and_stick", 
    #             size_factor=custom_vizs[pdb]["ligands"].get("size_factor", 0.6),
    #             custom={"molstar_representation_params": {"ignoreLight": True}})
    #         .color(
    #             custom={
    #                 "molstar_color_theme_name": "element-symbol",
    #                 "molstar_color_theme_params": {
    #                     "carbonColor": { "name": "uniform", "params": {"value": int("#666666".replace('#', ''), 16)} }
    #                 }
    #             }
    #         )
    #         # .opacity(opacity=0.9)
    #     )



    # Pockets
    pockets_strs = {}
    for i, pocket in enumerate(pockets):
        pocketf = Path(pocket["pocketsf"])
        name = pocketf.name
        if name not in assets:
            if "transformation" in custom_vizs[pdb]["camera"]:
                pockets_strs[name] = (
                    builder.download(url=name)
                    .parse(format="mmcif" if pocketf.suffix == ".cif" else "pdb")
                    .model_structure()
                    .transform(rotation=rot, translation=trans)
                )
            else:
                pockets_strs[name] = (
                    builder.download(url=name)
                    .parse(format="mmcif" if pocketf.suffix == ".cif" else "pdb")
                    .model_structure()
                )
            with open(pocketf, "r") as f:
                assets[name] = f.read().encode()
        
        (
            pockets_strs[name]
            .component(selector=[
                mvs.ComponentExpression(**sel)
                for sel in pocket["pocket_sel"]
            ])
            .representation(type="surface", custom={"molstar_representation_params": {"ignoreLight": True}})
            .color(color=pocket["color"])
            .opacity(
                opacity=(
                    custom_vizs[pdb]["pocket_opacity"]
                    .get(model, {pocket["pocketn"]: 0.5})
                    .get(pocket["pocketn"])
                )
            )
        )
    


    # View
    return mvs.MVSX(
        data=builder.get_state(),
        assets=assets
    )
    # v = mvsx.molstar_notebook(width= 1725, height= 875)
    # return v

In [172]:
def view_top(pdb, model, top=None):
    pocketf = models[model]["pocketf"]
    prob_key = models[model]["prob_key"]
    results = models[model]["results"][pdb]
    labelled = models[model]["labelled"].query(f"pdb == '{pdb}'").sort_values("prob", ascending=False)
    pos_pockets = labelled[labelled["label"] == 1]

    if top is None:
        top = labelled["label"].sum() or 1
    top_pockets = tuple(pocket for i, pocket in tuple(labelled.iterrows())[:top])

    for pocket in top_pockets:
        print(pocket["pocket"], {k: v for k, v in results[pocket["pocket"]].items() if k != "residues"}, "label:", pocket["label"])
        
    return view_pockets(
        pdb,
        model,
        tuple(
            [
                pocketf(pdb, pocket["pocket"], "green" if pocket["label"] == 1 else "blue")
                for pocket in top_pockets
            ] + [
                pocketf(pdb, pocket["pocket"], "orange")
                for i, pocket in pos_pockets.iterrows()
                if pocket["pocket"] not in (p["pocket"] for p in top_pockets)
            ]
        )
    )

# Viz

In [173]:
news.keys()

dict_keys(['8sgj', 'AF-A0A1D8PQM9-F1', '7l6r', '8vw5', '6yhr', '7xlq', '5b0u', '5uak', '4jqi'])

In [238]:
custom_vizs = {
    p: {
        "camera": {"camera": False}, # set a common camera after revision; also model-specific
        "ligands": {"none": False}, # set to True after revision; also model-specific; also size_factor
        "selections": {},
        "pocket_opacity": {}
    }
    for p in news.keys()
}


# custom_vizs["8sgj"].update({
#         "camera": {
#             "camera": """{
#     position: [95.49, 187.55, 251.44],
#     target: [152.01, 159.35, 167.46],
#     up: [0.84, 0.14, 0.52],
# }""",
#             "allositepro": """{
#     position: [223.05, 143.6, 75.27],
#     target: [152.01, 159.35, 167.46],
#     up: [0.8, 0.15, 0.59],
# }""",
#         },
#         "ligands": {
#             "none": True,
#             "size_factor": 0.7,
#             "model5": [{"auth_asym_id": "A", "auth_seq_id": 2404},]
#         },
#         "pocket_opacity": {
#             "allositepro": {"2": 0.7}
#         },
#         "selections": {
#             "protein_A": [
#                 {"auth_asym_id": "A", "beg_auth_seq_id": 1, "end_auth_seq_id": 176},
#                 {"auth_asym_id": "A", "beg_auth_seq_id": 283, "end_auth_seq_id": 494},
#             ]
#         }
#     })


custom_vizs["7l6r"].update({
    "camera": {
        "camera": """{
    position: [74.58, -29.57, 27.09],
    target: [92.25, 19.85, 25.6],
    up: [-0.18, 0.09, 0.98],
}""",
    },
    "ligands": {
        "none": True,
        # "size_factor": 0.7,
        "model5": [{"label_asym_id": "H"},],
        "allo": [{"label_asym_id": "H"},],
        "mefallosite": [{"label_asym_id": "H"},],
    },
    "selections": {
        "protein_A": [{"label_asym_id": "A", "beg_label_seq_id": 1, "end_label_seq_id": 295},]
    }
})

custom_vizs["5b0u"].update({
    "camera": {
        "camera": """{
    position: [42.49, -12.34, 85.5],
    target: [44.34, -49.6, 65.39],
    up: [0.88, -0.19, 0.43],
}""", # camera of original holo when a transformation is used
        "transformation": [-0.8731406674,0.4363838836,0.217242908,0,0.4454373001,0.5332115855,0.7192155565,0,0.1980176423,0.7247444454,-0.6599503786,0,46.1847527678,-119.2214202559,62.750732752,1],
    },
    "ligands": {"none": True,}
})


custom_vizs["8vw5"].update({
    "camera": {
        "camera": """{
    position: [31.03, 30.61, 94.01],
    target: [18.05, 10.31, 31.44],
    up: [0.18, 0.93, -0.34],
}""", # camera of original holo when a transformation is used
        "transformation": [0.5759028907,-0.6694201429,-0.4692680819,0,-0.7609016017,-0.6488147176,-0.0082592256,0,-0.298939146,0.361823347,-0.8830171304,0,26.5447597967,8.2235551791,46.6840095568,1],
    },
    "ligands": {"none": True,}
})


custom_vizs["6yhr"].update({
    "camera": {
        "camera": """{
    position: [77.37, 31.66, 32.78],
    target: [5.74, 27.52, 18.12],
    up: [0.04, 0.89, -0.46],
}""", # camera of original holo when a transformation is used
        "transformation": [0.1399118591,-0.9748662142,-0.1733797453,0,0.9696591642,0.1703479943,-0.1753358669,0,0.2004639046,-0.1435876918,0.969121663,0,20.2964648122,22.8520055729,-21.8875622483,1],
    },
    "ligands": {
        "none": True,
        # "size_factor": 0.7,
        "model5": [{"label_asym_id": "B"},],
        "allo": [{"label_asym_id": "B"},],
        "mefallosite": [{"label_asym_id": "B"},],
    },
    "selections": {
        "protein_A": [{"label_asym_id": "A", "beg_label_seq_id": 13, "end_label_seq_id": 430},]
    }
})

custom_vizs["5uak"].update({
    "camera": {
        "camera": """{
    position: [110.01, 224.39, 228.89],
    target: [150.45, 123.5, 140.12],
    up: [-0.62, 0.36, -0.69],
}""", # camera of original holo when a transformation is used
        "transformation": [-0.13105706571718778,-0.45247684104623354,-0.8820933929252743,0,-0.9802573751601144,0.1920238121179244,0.04714163790008763,0,0.1480524365547684,0.8708567987312964,-0.4687098378887251,0,292.49179494239024,39.73494877071238,344.4403539798741,1],
    },
    "ligands": {"none": True,},
    "pocket_opacity": {
        "model5": {"1": 0.35},
        "mefallosite": {"118": 0.65, "35": 0.65}
    },
})

custom_vizs["8sgj"].update({
    "camera": {
        "camera": """{
    position: [103.33, 108.95, 69.94],
    target: [130.69, 134.14, 126.6],
    up: [-0.91, 0.09, 0.4],
}""", # camera of original holo when a transformation is used
        "transformation": [-0.3653714725,-0.9250073087,-0.1042361066,0,0.3911907883,-0.2541901909,0.884509533,0,-0.8446735785,0.2823983458,0.4547281827,0,260.9762416101,255.1398289019,-59.2357144169,1],
    },
    "ligands": {"none": True,},
    "pocket_opacity": {
        "model5": {"1": 0.35},
        "allo": {"P_1": 0.4}
    },
})


custom_vizs["AF-A0A1D8PQM9-F1"].update({
    "camera": {
        "camera": """{
    position: [111.85, 20.63, 87.19],
    target: [47.48, 25.38, 18.04],
    up: [0.43, -0.78, -0.46],
}""", # camera of original holo when a transformation is used
        "transformation": [-0.2842114256,-0.769562924,0.5718363153,0,0.9520275233,-0.2970900437,0.0733559863,0,0.1134348286,0.5652525205,0.8170814695,0,46.6750136509,20.6831490134,16.2906627342,1],
    },
    "ligands": {"none": True,},
    "selections": {
        "protein_A": [{"label_asym_id": "A", "beg_label_seq_id": 213, "end_label_seq_id": 799},]
    }
})


custom_vizs["7xlq"].update({
    "camera": {
        "camera": """{
    position: [95.49, 187.55, 251.44],
    target: [152.01, 159.35, 167.46],
    up: [0.84, 0.14, 0.52],
}""", # camera of original holo when a transformation is used
        "model5": """{
    position: [143.51, 264.08, 162.99],
    target: [155.86, 159.89, 168.95],
    up: [0.85, 0.13, 0.51],
}""",
        "transformation": [0.9999010495,-0.0124626843,0.0065247819,0,0.0125041625,0.9999016232,-0.0063552928,0,-0.006444936,0.0064362509,0.9999585179,0,-1.1825644706,1.0517760401,-0.1924989214,1],
    },
    "ligands": {
        "none": True,
        "mefallosite": [{"label_asym_id": "Q"},],
    }
})


custom_vizs["4jqi"].update({
    "camera": {
        "camera": """{
    position: [224.61, 233.28, 104.7],
    target: [205.02, 209.34, 179.72],
    up: [-0.7, 0.71, 0.05],
}""", # camera of original holo when a transformation is used
        "transformation": [-0.9814851073,-0.0295621546,0.1892433968,0,-0.1332807051,0.8149966625,-0.5639296886,0,-0.1375617601,-0.5787110843,-0.8038471515,0,219.9594598634,211.6503674002,183.4590340108,1],
    },
    "ligands": {"none": True,},
    "pocket_opacity": {
        "mefallosite": {"1": 0.65, "8": 0.65}
    },
})


In [239]:
# custom_vizs["7yg5"].update({
#         "camera": {
#             "camera": """{
#     position: [95.49, 187.55, 251.44],
#     target: [152.01, 159.35, 167.46],
#     up: [0.84, 0.14, 0.52],
# }""",
#             "allositepro": """{
#     position: [223.05, 143.6, 75.27],
#     target: [152.01, 159.35, 167.46],
#     up: [0.8, 0.15, 0.59],
# }""",
#             "passer_ensemble": """{
#     position: [223.05, 143.6, 75.27],
#     target: [152.01, 159.35, 167.46],
#     up: [0.8, 0.15, 0.59],
# }""",
#         },
#         "ligands": {
#             "none": True,
#             "size_factor": 0.7,
#             "model5": [{"auth_asym_id": "A", "auth_seq_id": 2404},]
#         },
#         "pocket_opacity": {
#             "allositepro": {"2": 0.7}
#         }
#     })


# custom_vizs["7gqu"].update({
#         "camera": {
#             "camera": """{
#     position: [77.37, 31.66, 32.78],
#     target: [5.74, 27.52, 18.12],
#     up: [0.04, 0.89, -0.46],
# }""",
#         },
#         "ligands": {"none": True},
#         "pocket_opacity": {
#             "allositepro": {"2": 0.7}
#         }
#     })



# custom_vizs["8aq6"].update({
#         "camera": {
#             "camera": """{
#     position: [42.49, -12.34, 85.5],
#     target: [44.34, -49.6, 65.39],
#     up: [0.88, -0.19, 0.43],
# }""",
#         },
#         "ligands": {"none": True},
#         "pocket_opacity": {
#             "allositepro": {"1": 0.8},
#             "passer_ensemble": {"3": 0.8},
#             "allo": {"P_2": 0.8}
#         }
#     })



# custom_vizs["8jp0"].update({
#         "camera": {
#             "camera": """{
#     position: [103.33, 108.95, 69.94],
#     target: [130.69, 134.14, 126.6],
#     up: [-0.91, 0.09, 0.4],
# }""", # Establish the camera
#         },
#         "ligands": {"none": True},
#         "pocket_opacity": {
#             "allositepro": {"2": 0.6}
#         }
#     })



# custom_vizs["8qni"].update({
#         "camera": {
#             "camera": """{
#     position: [31.03, 30.61, 94.01],
#     target: [18.05, 10.31, 31.44],
#     up: [0.18, 0.93, -0.34],
# }""", # Establish the camera
#         },
#         "ligands": {"none": True},
#     })


# custom_vizs["8uk6"].update({
#         "camera": {
#             "camera": """{
#     position: [111.85, 20.63, 87.19],
#     target: [47.48, 25.38, 18.04],
#     up: [0.43, -0.78, -0.46],
# }""", # Establish the camera
#         },
#         "ligands": {"none": True},
#     })




# custom_vizs["8v81"].update({
#         "camera": {
#             "camera": """{
#     position: [110.01, 224.39, 228.89],
#     target: [150.45, 123.5, 140.12],
#     up: [-0.62, 0.36, -0.69],
# }""", # Establish the camera
#         },
#         "ligands": {
#             "none": True, 
#             "size_factor": 0.7,
#             "passer_ensemble": [{"auth_asym_id": "A", "beg_auth_seq_id": 1501, "end_auth_seq_id": 1504},]
#         },
#         "pocket_opacity": {
#             "allositepro": {"1": 0.7},
#             # "passer_ensemble": {"49": 0.6}
#         }
#     })



# custom_vizs["9dnm"].update({
#         "camera": {
#             "camera": """{
#     position: [224.61, 233.28, 104.7],
#     target: [205.02, 209.34, 179.72],
#     up: [-0.7, 0.71, 0.05],
# }""", # Establish the camera
#         },
#         "ligands": {
#             "none": True, 
#             "size_factor": 0.7
#         },
#         "selections": {
#             "protein_A": [
#                 {"auth_asym_id": "A", "beg_auth_seq_id": 1, "end_auth_seq_id": 176},
#                 {"auth_asym_id": "A", "beg_auth_seq_id": 283, "end_auth_seq_id": 494},
#             ]
#         },
#         "pocket_opacity": {
#             "passer_ensemble": {"20": 0.7},
#             "allo": {"P_0": 0.5}
#         }
#     })

In [240]:
news.keys()

dict_keys(['8sgj', 'AF-A0A1D8PQM9-F1', '7l6r', '8vw5', '6yhr', '7xlq', '5b0u', '5uak', '4jqi'])

In [241]:
modell = ["model5", "mefallosite", "allo",]#"allositepro",

In [242]:
i = 0

In [245]:
curr_model = modell[i]; print(curr_model)
i += 1
mvsx = view_top('4jqi', curr_model); mvsx.molstar_notebook(width= 1100, height= 500)#(width= 1500, height= 600)#

allo
P_0 {'pred': 1, 'prob': 0.023459999999999998} label: 0


<IPython.core.display.Javascript object>

In [1]:
mvsx.data.dict()

## Ignorelight tests